# E178 Adversarial Learning: Fraud Detection Classifier Assessment

In this performance assessment you act as an AI red team and defender for a credit card fraud detection classifier. You will carry the classifier through the complete adversarial machine learning lifecycle:

| Task | Part | What you will do |
|---|---|---|
| **Task 1** | Part 1: Threat model | Develop an adversarial threat model for the classifier using MITRE ATLAS as a reference framework |
| | Part 2: Attacks | Execute poisoning (backdoor and clean-label), evasion, gray-box, black-box, and extraction attacks, evaluate their effectiveness, and connect the results back to your threat model |
| **Task 2** | Part 1: Defenses | Implement adversarial training and a second defense mechanism, compare performance before and after, and recommend a justified defense configuration |
| | Part 2: Privacy | Conduct membership inference and model inversion testing, analyze the privacy implications, and produce an adversarial evaluation report for technical and non-technical audiences |

**Tooling:** The Adversarial Robustness Toolbox (ART), Foolbox, PyTorch, scikit-learn, pandas, and Matplotlib are preinstalled in the **Python (Adversarial Learning)** kernel. MITRE ATLAS is available at <https://atlas.mitre.org/>.

### How to use this notebook

* Run the **Setup** section first. It loads the dataset, creates the data partitions, and loads the baseline classifier that is the system under test.
* Work through the tasks in order. Later tasks build on earlier ones; for example, your defenses should target the highest risks from your threat model, and your privacy testing should include your recommended defense configuration.
* Cells marked `# Your code here` are for your implementation. Add as many code cells as you need.
* Cells marked **Your response** are for your written analysis. Support every claim with evidence produced in this notebook (metrics, tables, or figures).
* Save figures with `save_fig(fig, "name")` and any tables you produce to the `reports/` folder so you can include them as supporting evidence.

Every experiment in this assessment can run on the lab's CPU notebook instance. Training the baseline architecture once takes seconds; adversarial training and query-based attacks take longer, so start with small budgets and increase them once your code works.

## 0. Setup

### 0.1 Imports and configuration

Import any additional ART, Foolbox, or scikit-learn classes you need in the cells where you use them.

In [ ]:
import hashlib
import json
import logging
import os
import time
import warnings
from collections import OrderedDict
from pathlib import Path

os.environ.setdefault("TQDM_DISABLE", "1")  # hide ART progress bars
warnings.filterwarnings("ignore")
logging.getLogger("art").setLevel(logging.ERROR)

import art
import foolbox as fb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import torch
import torch.nn as nn
from IPython.display import Markdown, display
from sklearn.metrics import average_precision_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from art.estimators.classification import PyTorchClassifier

pd.set_option("display.precision", 3)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

for name, module in [("numpy", np), ("pandas", pd), ("scikit-learn", sklearn), ("torch", torch),
                     ("adversarial-robustness-toolbox", art), ("foolbox", fb)]:
    print(f"{name:32s} {module.__version__}")

In [ ]:
SEED = 178

DATA_PATH = Path("datasets/creditcard_fraud_sample.csv.gz")
MODEL_PATH = Path("models/baseline_fraud_mlp.pt")
MODEL_CARD_PATH = Path("models/baseline_fraud_mlp.json")
REPORT_DIR = Path("reports")
FIG_DIR = REPORT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Baseline model hyperparameters
HIDDEN_LAYERS = (64, 32)
FRAUD_CLASS_WEIGHT = 5.0  # up-weights the rare fraud class in the loss
EPOCHS = 30
BATCH_SIZE = 256
LEARNING_RATE = 1e-3

LEGIT, FRAUD = 0, 1
RESULTS = {}  # store key metrics here as you go; saved to reports/results.json at the end


def seed_everything(seed=SEED):
    np.random.seed(seed)
    torch.manual_seed(seed)


def save_fig(fig, name):
    fig.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")


seed_everything()

### 0.2 The fraud detection dataset

The lab uses a sample of the **ULB Credit Card Fraud Detection** dataset: card transactions made by European cardholders over two days in September 2013. To protect cardholder confidentiality, the bank released 28 features (`V1`–`V28`) as anonymized principal components, plus the transaction `Time` and `Amount`. `Class` is `1` for fraud.

The sample (`datasets/creditcard_fraud_sample.csv.gz`) contains every unique fraud transaction plus 24,000 randomly sampled legitimate ones, so it keeps a realistic class imbalance of about 2% fraud. See `datasets/datasets.md` for details. The model uses `V1`–`V28` and the log of `Amount` as its 29 input features.

In [ ]:
df = pd.read_csv(DATA_PATH)
df["log_amount"] = np.log1p(df["Amount"])

FEATURES = [f"V{i}" for i in range(1, 29)] + ["log_amount"]
N_FEATURES = len(FEATURES)

print(f"Transactions: {len(df):,}  |  features used: {N_FEATURES}")
print(f"Fraud rate:   {df['Class'].mean():.2%}")
display(df["Class"].value_counts().rename({0: "legitimate", 1: "fraud"}).to_frame("count"))
display(df.head())

### 0.3 Data partitions

Three disjoint partitions model who has access to what:

| Partition | Variables | Share | Role |
|---|---|---|---|
| `train` | `X_train`, `y_train` | 60% | The bank's training data. These are the **members** for membership inference. |
| `test` | `X_test`, `y_test` | 20% | Held-out evaluation data, and **non-members** for membership inference. |
| `attacker` | `X_att`, `y_att` | 20% | Data from the same distribution that an attacker could plausibly obtain (their own transactions, purchased or breached data). |

Features are standardized with statistics from the training partition only, so perturbation budgets are in **standard deviations** of each feature. `CLIP_MIN` and `CLIP_MAX` hold the range of each feature across all real transactions; the ART classifiers built by `wrap_model` clip inputs to this range.

`X_train_raw`, `X_test_raw`, and `X_att_raw` hold the unstandardized values, and `scaler.inverse_transform` converts standardized values back.

In [ ]:
X_all = df[FEATURES].to_numpy(np.float32)
y_all = df["Class"].to_numpy()

X_rest_raw, X_test_raw, y_rest, y_test = train_test_split(
    X_all, y_all, test_size=0.20, stratify=y_all, random_state=SEED)
X_train_raw, X_att_raw, y_train, y_att = train_test_split(
    X_rest_raw, y_rest, test_size=0.25, stratify=y_rest, random_state=SEED)

scaler = StandardScaler().fit(X_train_raw)
X_train, X_test, X_att = (scaler.transform(a).astype(np.float32)
                          for a in (X_train_raw, X_test_raw, X_att_raw))

# Feasible feature range
X_known = np.vstack([X_train, X_test, X_att])
CLIP_MIN, CLIP_MAX = X_known.min(axis=0), X_known.max(axis=0)

display(pd.DataFrame({
    "rows": [len(y_train), len(y_test), len(y_att)],
    "fraud": [y_train.sum(), y_test.sum(), y_att.sum()],
    "fraud rate": [y_train.mean(), y_test.mean(), y_att.mean()],
}, index=["train", "test", "attacker"]))

### 0.4 Baseline fraud classifier

The system under test is a multilayer perceptron (29 → 64 → 32 → 2, ReLU) trained with a class-weighted cross-entropy loss. The trained weights and a model card are in `models/` (see `models/models.md`). The loading cell verifies the weights' SHA-256 hash against the model card before loading them.

The helper functions below are available throughout the notebook:

| Helper | Purpose |
|---|---|
| `build_model(hidden)` | Create an untrained PyTorch MLP with the given hidden layer sizes |
| `wrap_model(model)` | Wrap a PyTorch model in an ART `PyTorchClassifier` with the baseline's loss, optimizer, and clip values |
| `train_classifier(x, y, hidden, epochs, seed)` | Train a new classifier with the baseline's settings and return it as an ART classifier |
| `onehot(labels)` | Convert integer labels to the one-hot format ART expects |
| `fraud_probability(clf, x)` | Fraud probability for each row |
| `predict_label(clf, x)` | Predicted label (`LEGIT` = 0, `FRAUD` = 1) for each row |
| `evaluate(clf, x, y)` | Fraud-class precision, recall, F1, ROC-AUC, and PR-AUC (defaults to the test partition) |

`baseline` is the loaded model, wrapped as an ART `PyTorchClassifier`; `baseline.model` is the underlying PyTorch module.

In [ ]:
def onehot(labels):
    return np.eye(2, dtype=np.float32)[np.asarray(labels, dtype=int)]


def build_model(hidden=HIDDEN_LAYERS):
    layers, width = [], N_FEATURES
    for units in hidden:
        layers += [nn.Linear(width, units), nn.ReLU()]
        width = units
    return nn.Sequential(OrderedDict(hidden=nn.Sequential(*layers), output=nn.Linear(width, 2)))


def wrap_model(model, fraud_weight=FRAUD_CLASS_WEIGHT, lr=LEARNING_RATE):
    loss = nn.CrossEntropyLoss(weight=torch.tensor([1.0, fraud_weight]))
    return PyTorchClassifier(
        model=model,
        loss=loss,
        optimizer=torch.optim.Adam(model.parameters(), lr=lr),
        input_shape=(N_FEATURES,),
        nb_classes=2,
        clip_values=(CLIP_MIN, CLIP_MAX),
    )


def train_classifier(x, y, hidden=HIDDEN_LAYERS, epochs=EPOCHS, seed=SEED):
    seed_everything(seed)
    clf = wrap_model(build_model(hidden))
    clf.fit(x, onehot(y), batch_size=BATCH_SIZE, nb_epochs=epochs, verbose=False)
    return clf


def fraud_probability(clf, x):
    return torch.softmax(torch.as_tensor(clf.predict(x)), dim=1)[:, FRAUD].numpy()


def predict_label(clf, x):
    return clf.predict(x).argmax(axis=1)


def evaluate(clf, x=None, y=None):
    x = X_test if x is None else x
    y = y_test if y is None else y
    prob = fraud_probability(clf, x)
    pred = (prob >= 0.5).astype(int)
    return {
        "precision": precision_score(y, pred, zero_division=0),
        "recall": recall_score(y, pred),
        "f1": f1_score(y, pred),
        "roc_auc": roc_auc_score(y, prob),
        "pr_auc": average_precision_score(y, prob),
    }


def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

In [ ]:
model_card = json.loads(MODEL_CARD_PATH.read_text())
if sha256(MODEL_PATH) != model_card["sha256"]:
    raise RuntimeError("Model file hash does not match the model card. Refusing to load it.")
if not np.allclose(model_card["scaler_mean"], scaler.mean_, atol=1e-5):
    raise RuntimeError("Preprocessing does not match the model card. Check the dataset and SEED.")

model = build_model(tuple(model_card["hidden_layers"]))
model.load_state_dict(torch.load(MODEL_PATH, weights_only=True))
baseline = wrap_model(model)
print(f"Loaded baseline model (sha256 {model_card['sha256'][:12]}...) - hash verified")

RESULTS["baseline"] = evaluate(baseline)
display(pd.DataFrame(RESULTS["baseline"], index=["baseline (test)"]))

---
## Task 1, Part 1: Adversarial threat model

### System description

The bank uses the classifier to score every card-not-present transaction in real time through an internal scoring API. Transactions scored as fraud are declined or held for an analyst; customers and merchants see whether a transaction was approved or declined. The model is retrained weekly on recent transactions. Training labels come from chargebacks and analyst decisions, and transactions that are never reported as fraud are labeled legitimate. A partner program lets approved merchants query a scoring endpoint that returns a fraud score for a transaction. The model weights, training code, and training data are stored in the bank's internal model registry and data pipeline.

### Requirements

Develop an adversarial threat model for this classifier using MITRE ATLAS as a reference framework. Your threat model must:

1. Identify the assets the system needs to protect and the adversaries who might target it, including each adversary's goals, capabilities, and knowledge.
2. Identify the entry points through which an adversary can interact with or influence the model.
3. Describe the attack paths each adversary could follow, mapped to MITRE ATLAS tactics and techniques (with technique IDs).
4. Assess the likelihood and impact of each attack path on a defined scale, and give a rationale for each assessment.

Include, at minimum, an attack path for every attack you execute in Task 1 Part 2 and Task 2 Part 2, so that you can connect your results back to the threat model.

**Your response:** Describe the assets, adversaries (goals, capabilities, knowledge), and entry points.

_Double-click this cell to write your response._

Record your attack paths as a table. The template below saves it to `reports/threat_model.csv`; adjust the columns if you need to.

In [ ]:
threat_model = pd.DataFrame(
    [
        # One dict per attack path, for example:
        # {"id": "T1", "attack path": "...", "adversary": "...", "goal": "...", "entry points": "...",
        #  "ATLAS techniques": "...", "likelihood": ..., "impact": ..., "rationale": "..."},
    ],
    columns=["id", "attack path", "adversary", "goal", "entry points", "ATLAS techniques",
             "likelihood", "impact", "rationale"],
)
threat_model.to_csv(REPORT_DIR / "threat_model.csv", index=False)
display(threat_model)

**Your response:** Explain your likelihood and impact scale, and justify the highest-risk attack paths.

_Double-click this cell to write your response._

---
## Task 1, Part 2: Attack execution and effectiveness

Execute each attack below against the baseline classifier using ART and/or Foolbox. Relevant ART modules include `art.attacks.poisoning`, `art.attacks.evasion`, and `art.attacks.extraction`; Foolbox attacks are in `foolbox.attacks`.

### Requirements for every attack

* State which attack path in your threat model the attack tests, and the attacker's knowledge and access. Make sure your code only uses what that attacker would have; for example, a black-box attacker can only observe the model's outputs.
* Use the `attacker` partition (`X_att`, `y_att`) for any data the attacker controls or holds. Evaluate on the `test` partition and never train an attacker's model on it.
* Define how you measure attack effectiveness, and use measures that are meaningful for an imbalanced fraud dataset. Use consistent measures so that attacks can be compared.
* Report the attack's cost alongside its success, for example the perturbation budget, the number of poisoned rows, or the number of model queries.
* Show how effectiveness changes as you vary at least one key attack parameter.
* Analyze the results and explain what they mean for the fraud detection system.

### 1.2.1 Backdoor (dirty-label) poisoning

Poison the training data so that the retrained model learns a hidden trigger. Report the attack's effect on transactions that carry the trigger **and** on the model's performance on clean data.

In [ ]:
# Your code here

**Your response:** Analyze the effectiveness and stealth of the backdoor poisoning attack.

_Double-click this cell to write your response._

### 1.2.2 Clean-label poisoning

Poison the training data without changing any labels: every poisoned row must keep its correct label. Compare the result with the dirty-label attack.

In [ ]:
# Your code here

**Your response:** Analyze the effectiveness and stealth of the clean-label poisoning attack, and compare it with the dirty-label attack.

_Double-click this cell to write your response._

### 1.2.3 Evasion

Craft adversarial examples that cause fraudulent transactions to be classified as legitimate, using full (white-box) access to the baseline model. Compare at least two evasion attacks or attack configurations.

In [ ]:
# Your code here

**Your response:** Analyze the effectiveness of the evasion attacks.

_Double-click this cell to write your response._

### 1.2.4 Gray-box attack

Attack the baseline with partial knowledge only. Define what the attacker knows and does not know, and do not use the baseline's weights or gradients in the attack.

In [ ]:
# Your code here

**Your response:** Analyze the effectiveness of the gray-box attack, and compare it with white-box evasion.

_Double-click this cell to write your response._

### 1.2.5 Black-box attack

Attack the baseline using only its outputs. Define which output the attacker observes (for example the approve/decline decision or the partner fraud score), and record the number of queries the attack makes.

In [ ]:
# Your code here

**Your response:** Analyze the effectiveness and cost of the black-box attack.

_Double-click this cell to write your response._

### 1.2.6 Model extraction

Use queries to the baseline to train a copy of it. Measure how closely the copy matches the baseline and how the result depends on the number of queries.

In [ ]:
# Your code here

**Your response:** Analyze the effectiveness of the extraction attack and what an attacker could do with the copy.

_Double-click this cell to write your response._

### 1.2.7 Attack effectiveness summary

Summarize the results of every attack in one table that links each attack to its attack path in your threat model, and save it to `reports/attack_summary.csv`. Then use the evidence to revisit your threat model: state whether each likelihood and impact assessment still holds, and update the ones that do not.

In [ ]:
# Your code here

**Your response:** Connect your attack results back to your threat model, and explain any changes to your likelihood and impact assessments.

_Double-click this cell to write your response._

---
## Task 2, Part 1: Defense implementation and evaluation

Implement two defense mechanisms against the attacks from Task 1:

1. **Adversarial training.** ART's trainers are in `art.defences.trainer`.
2. **A second defense of your choice** that targets a different threat from your threat model, for example a poisoning detector or training-data sanitization, an input preprocessor, an output postprocessor, or another technique. Relevant ART modules include `art.defences.detector`, `art.defences.preprocessor`, and `art.defences.postprocessor`.

### Requirements

* Explain which attack paths from your threat model each defense targets, and why you chose it.
* For each defense, compare model performance on clean data **and** attack success before and after the defense, using the same measures as Task 1 Part 2.
* Evaluate the defenses in combination as well as individually.
* Report the cost of each defense, for example training time or the effect on precision and recall.
* Recommend a defense configuration and justify it with your results and with deployment considerations for the fraud detection system.

Save your before/after comparison table to `reports/defense_comparison.csv`.

### 2.1.1 Adversarial training

In [ ]:
# Your code here

**Your response:** Analyze the effect of adversarial training on model performance and attack success.

_Double-click this cell to write your response._

### 2.1.2 Second defense

In [ ]:
# Your code here

**Your response:** Describe your second defense and analyze its effect on model performance and attack success.

_Double-click this cell to write your response._

### 2.1.3 Combined evaluation and recommendation

In [ ]:
# Your code here

**Your response:** Recommend a defense configuration and justify it using your results and deployment considerations.

_Double-click this cell to write your response._

---
## Task 2, Part 2: Privacy testing

Test what the classifier reveals about the people in its training data. Relevant ART modules are `art.attacks.inference.membership_inference` and `art.attacks.inference.model_inversion`. The `train` partition holds the members and the `test` partition holds the non-members.

### Requirements

* Conduct membership inference testing and model inversion testing against the baseline **and** your recommended defense configuration.
* Explain the attacker's access and knowledge for each test, and link each test to its attack path in your threat model.
* Report results with measures that make clear how much better than chance each attack performs.
* Analyze the privacy implications of your results, including any differences between groups of records and any effects of your defenses on privacy.

### 2.2.1 Membership inference

In [ ]:
# Your code here

**Your response:** Analyze the membership inference results.

_Double-click this cell to write your response._

### 2.2.2 Model inversion

In [ ]:
# Your code here

**Your response:** Analyze the model inversion results.

_Double-click this cell to write your response._

### 2.2.3 Privacy implications

**Your response:** Summarize the privacy implications of your findings and justify them with evidence from your attack, defense, membership inference, and model inversion testing. Recommend controls for the risks you identify.

_Double-click this cell to write your response._

---
## Adversarial evaluation report

Write an adversarial evaluation report that synthesizes your findings from both tasks. It must include:

* A **technical summary** for the security and data science teams: the system under test, your threat model's highest risks, attack results, defense results and recommendation, privacy results, and the limitations of your evaluation.
* An **executive summary** for non-technical stakeholders: the key risks, what they mean for the business and its customers, and what you recommend, in plain language.

**Your response:** Technical summary.

_Double-click this cell to write your response._

**Your response:** Executive summary (non-technical).

_Double-click this cell to write your response._

### Save supporting evidence

Run the cell below to save the metrics you stored in `RESULTS` to `reports/results.json` and list the evidence files in `reports/`.

In [ ]:
def to_builtin(value):
    if isinstance(value, dict):
        return {str(k): to_builtin(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_builtin(v) for v in value]
    if isinstance(value, (np.floating, np.integer)):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    return value


(REPORT_DIR / "results.json").write_text(json.dumps(to_builtin(RESULTS), indent=2, default=str))
print("Saved:", *sorted(str(p) for p in REPORT_DIR.rglob("*") if p.is_file()), sep="\n  ")

### Submission checklist

* Run every cell from top to bottom (**Kernel > Restart Kernel and Run All Cells**) so that all outputs are present.
* Save the executed notebook and export a copy (**File > Save and Export Notebook As > HTML**).
* Download the `reports/` folder, including `reports/figures/`, as supporting evidence.
* Confirm your submission covers the threat model, the attack execution and effectiveness analysis, the defense implementation and recommendation, and the privacy implications brief and adversarial evaluation report.